# 04 — Model Export

Three outputs:
1. **tzniut.tflite** (Android) — INT8 quantized
2. **tzniut.mlpackage** (iOS / macOS) — INT8 weight quantization
3. **tzniut_full.pt** (full PyTorch) — for future fine-tuning

In [ ]:
import os, pathlib
if not pathlib.Path('zahava-tzniut').exists():
    !git clone https://github.com/zahava-networks/zahava-tzniut.git
os.chdir('zahava-tzniut')
!pip install -q -r requirements.txt timm torch torchvision ai-edge-torch coremltools

In [ ]:
# Build the INT8 calibration set
from pipelines.export.calibration_set import build as build_calib
build_calib()

In [ ]:
# Export TFLite
from pipelines.export.export_tflite import export as export_tflite
export_tflite()

In [ ]:
# Export CoreML
from pipelines.export.export_coreml import export as export_coreml
export_coreml()

In [ ]:
# Export full PyTorch checkpoint
from pipelines.export.export_full import export as export_full
export_full()

In [ ]:
# Push to HF model repo
from huggingface_hub import HfApi, upload_folder, upload_file
from pipelines.common import require_env
api = HfApi(token=require_env('HF_TOKEN'))
repo = require_env('HF_MODEL_REPO')
api.create_repo(repo, exist_ok=True)
for f in ['models/tzniut.tflite', 'models/tzniut.tflite.json',
          'models/tzniut_full.pt', 'models/tzniut_full.json',
          'models/calibration_set.npy', 'config/thresholds.yaml']:
    if pathlib.Path(f).exists():
        upload_file(path_or_fileobj=f, path_in_repo=f.split('/')[-1], repo_id=repo)
upload_folder(folder_path='models/tzniut.mlpackage', path_in_repo='tzniut.mlpackage', repo_id=repo)
print(f'pushed to https://huggingface.co/{repo}')